# Assignment 1 — Association Between Dependent and Independent Variables
**Dataset:** UCI Student Performance (`student-por.csv`, Portuguese language course)  
**Independent variables:** `studytime`, `failures`, `absences`  
**Dependent variable:** `G3` (final grade, 0–20)

**Aim:** Identify and quantify the association between study-related inputs (study time, past failures, absences) and the final grade G3 using descriptive statistics, Pearson/Spearman correlations, heatmaps, scatter plots and regression trend lines.


## 1. Library imports

In [1]:
import io
import os
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

os.makedirs("outputs", exist_ok=True)
print("Libraries imported. random_state =", RANDOM_STATE)

Matplotlib is building the font cache; this may take a moment.


Libraries imported. random_state = 42


## 2. Dataset loading (reproducible, from official UCI source)

In [2]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"

def load_student_por(url=UCI_URL):
    print(f"Downloading: {url}")
    raw = urllib.request.urlopen(url, timeout=60).read()
    zf = zipfile.ZipFile(io.BytesIO(raw))
    print("Archive contents:", zf.namelist())
    with zf.open("student-por.csv") as f:
        df = pd.read_csv(f, sep=";")
    return df

df = load_student_por()
print("Shape:", df.shape)
print("Columns:", list(df.columns))

Downloading: https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip


Archive contents: ['student-mat.csv', 'student-por.csv', 'student-merge.R', 'student.txt']
Shape: (649, 33)
Columns: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


## 3. Dataset preview

In [3]:
df.head(10)

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,0,yes,no,no,no,yes,yes,yes,no,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,no,yes,yes,yes,yes,yes,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,no,no,yes,yes,no,no,4,3,2,1,2,5,0,11,13,13
5,GP,M,16,U,LE3,T,4,3,services,other,reputation,mother,1,2,0,no,yes,no,yes,yes,yes,yes,no,5,4,2,1,2,5,6,12,12,13
6,GP,M,16,U,LE3,T,2,2,other,other,home,mother,1,2,0,no,no,no,no,yes,yes,yes,no,4,4,4,1,1,3,0,13,12,13
7,GP,F,17,U,GT3,A,4,4,other,teacher,home,mother,2,2,0,yes,yes,no,no,yes,yes,no,no,4,1,4,1,1,1,2,10,13,13
8,GP,M,15,U,LE3,A,3,2,services,other,home,mother,1,2,0,no,yes,no,no,yes,yes,yes,no,4,2,2,1,1,1,0,15,16,17
9,GP,M,15,U,GT3,T,3,4,other,other,home,mother,1,2,0,no,yes,no,yes,yes,yes,yes,no,5,5,1,1,1,5,0,12,12,13


In [4]:
print("dtypes:"); print(df.dtypes)
print("\nMissing values per column (total):", int(df.isnull().sum().sum()))
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().sum() > 0 else "No missing values.")
print("\nRows:", len(df))

dtypes:
school          str
sex             str
age           int64
address         str
famsize         str
Pstatus         str
Medu          int64
Fedu          int64
Mjob            str
Fjob            str
reason          str
guardian        str
traveltime    int64
studytime     int64
failures      int64
schoolsup       str
famsup          str
paid            str
activities      str
nursery         str
higher          str
internet        str
romantic        str
famrel        int64
freetime      int64
goout         int64
Dalc          int64
Walc          int64
health        int64
absences      int64
G1            int64
G2            int64
G3            int64
dtype: object

Missing values per column (total): 0
No missing values.

Rows: 649


## 4. Data preprocessing / selection for this assignment

In [5]:
indep = ["studytime", "failures", "absences"]
dep = "G3"
cols = indep + [dep]

# Validate expected columns exist
assert all(c in df.columns for c in cols), f"Missing expected columns: {cols}"
sub = df[cols].copy()
print("Selected columns:", cols)
print("Any NaN in selection:", int(sub.isnull().sum().sum()))
print(sub.describe().T)

Selected columns: ['studytime', 'failures', 'absences', 'G3']
Any NaN in selection: 0
           count       mean       std  min   25%   50%   75%   max
studytime  649.0   1.930663  0.829510  1.0   1.0   2.0   2.0   4.0
failures   649.0   0.221880  0.593235  0.0   0.0   0.0   0.0   3.0
absences   649.0   3.659476  4.640759  0.0   0.0   2.0   6.0  32.0
G3         649.0  11.906009  3.230656  0.0  10.0  12.0  14.0  19.0


## 5. Descriptive statistics

In [6]:
desc = sub.describe(percentiles=[0.25, 0.5, 0.75]).T
desc["variance"] = sub.var()
desc["skew"] = sub.skew()
desc["kurtosis"] = sub.kurt()
print(desc.round(4))
desc.to_csv("outputs/descriptive_statistics.csv")
print("\nSaved: outputs/descriptive_statistics.csv")

           count     mean     std  min   25%   50%   75%   max  variance  \
studytime  649.0   1.9307  0.8295  1.0   1.0   2.0   2.0   4.0    0.6881   
failures   649.0   0.2219  0.5932  0.0   0.0   0.0   0.0   3.0    0.3519   
absences   649.0   3.6595  4.6408  0.0   0.0   2.0   6.0  32.0   21.5366   
G3         649.0  11.9060  3.2307  0.0  10.0  12.0  14.0  19.0   10.4371   

             skew  kurtosis  
studytime  0.6996    0.0378  
failures   3.0927    9.8244  
absences   2.0207    5.7811  
G3        -0.9129    2.7122  

Saved: outputs/descriptive_statistics.csv


## 6. Pearson and Spearman correlations with G3

In [7]:
results = []
for col in indep:
    x = sub[col].to_numpy()
    y = sub[dep].to_numpy()
    pr, pp = stats.pearsonr(x, y)
    sr, sp = stats.spearmanr(x, y)
    results.append({"feature": col, "pearson_r": pr, "pearson_p": pp,
                    "spearman_rho": sr, "spearman_p": sp})
corr_df = pd.DataFrame(results).set_index("feature")
print(corr_df.round(4))

print("\nFull Pearson correlation matrix:")
print(sub.corr(method="pearson").round(4))
print("\nFull Spearman correlation matrix:")
print(sub.corr(method="spearman").round(4))
corr_df.to_csv("outputs/pearson_spearman_vs_G3.csv")

           pearson_r  pearson_p  spearman_rho  spearman_p
feature                                                  
studytime     0.2498     0.0000        0.2747         0.0
failures     -0.3933     0.0000       -0.4484         0.0
absences     -0.0914     0.0199       -0.1585         0.0

Full Pearson correlation matrix:
           studytime  failures  absences      G3
studytime     1.0000   -0.1474   -0.1184  0.2498
failures     -0.1474    1.0000    0.1228 -0.3933
absences     -0.1184    0.1228    1.0000 -0.0914
G3            0.2498   -0.3933   -0.0914  1.0000

Full Spearman correlation matrix:
           studytime  failures  absences      G3
studytime     1.0000   -0.1603   -0.1169  0.2747
failures     -0.1603    1.0000    0.1209 -0.4484
absences     -0.1169    0.1209    1.0000 -0.1585
G3            0.2747   -0.4484   -0.1585  1.0000


## 7. Correlation heatmap

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(sub.corr(method="pearson").round(3), annot=True, cmap="coolwarm",
            vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Pearson correlation")
sns.heatmap(sub.corr(method="spearman").round(3), annot=True, cmap="coolwarm",
            vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Spearman correlation")
plt.suptitle("Correlation heatmaps: studytime / failures / absences / G3")
plt.tight_layout()
plt.savefig("outputs/correlation_heatmap.png", dpi=150)
plt.close()
print("Saved: outputs/correlation_heatmap.png")

Saved: outputs/correlation_heatmap.png


## 8. Scatter plots with regression trend lines

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, col in zip(axes, indep):
    sns.regplot(x=col, y=dep, data=sub, ax=ax,
                scatter_kws={"alpha": 0.45, "s": 18}, line_kws={"color": "red"})
    ax.set_title(f"{col} vs {dep}")
    ax.set_xlabel(col)
axes[0].set_ylabel("G3 (final grade)")
plt.suptitle("Scatter plots with linear trend lines")
plt.tight_layout()
plt.savefig("outputs/scatter_trends.png", dpi=150)
plt.close()
print("Saved: outputs/scatter_trends.png")

Saved: outputs/scatter_trends.png


## 9. Regression trend-line slopes (OLS, one feature at a time)

In [10]:
from sklearn.linear_model import LinearRegression

for col in indep:
    X = sub[[col]].to_numpy()
    y = sub[dep].to_numpy()
    m = LinearRegression().fit(X, y)
    print(f"{col}: slope={m.coef_[0]:.4f}, intercept={m.intercept_:.4f}, R^2={m.score(X, y):.4f}")

studytime: slope=0.9728, intercept=10.0278, R^2=0.0624
failures: slope=-2.1419, intercept=12.3813, R^2=0.1547
absences: slope=-0.0636, intercept=12.1388, R^2=0.0084


## 10. Interpretation (computed, not hand-written)

In [11]:
for _, row in corr_df.iterrows():
    feat = row.name
    def strength(a):
        a = abs(a)
        if a < 0.10: return "negligible"
        if a < 0.30: return "weak"
        if a < 0.50: return "moderate"
        return "strong"
    print(f"{feat} vs G3: Pearson r={row['pearson_r']:.3f} (p={row['pearson_p']:.3g}, {strength(row['pearson_r'])}), "
          f"Spearman rho={row['spearman_rho']:.3f} (p={row['spearman_p']:.3g}, {strength(row['spearman_rho'])})")
print()
print("studytime is ordinal 1(<2h)..4(>10h); failures counts past class failures; absences counts school absences.")
print("Expectation: studytime positively associated with G3; failures and absences negatively associated.")
print("Exact magnitudes/significance are printed above and in outputs/pearson_spearman_vs_G3.csv.")

studytime vs G3: Pearson r=0.250 (p=1.09e-10, weak), Spearman rho=0.275 (p=1.06e-12, weak)
failures vs G3: Pearson r=-0.393 (p=1.93e-25, moderate), Spearman rho=-0.448 (p=2.05e-33, moderate)
absences vs G3: Pearson r=-0.091 (p=0.0199, negligible), Spearman rho=-0.159 (p=4.99e-05, weak)

studytime is ordinal 1(<2h)..4(>10h); failures counts past class failures; absences counts school absences.
Expectation: studytime positively associated with G3; failures and absences negatively associated.
Exact magnitudes/significance are printed above and in outputs/pearson_spearman_vs_G3.csv.


## 11. Result and conclusion
- Descriptive statistics, Pearson/Spearman correlations, heatmaps, scatter plots and OLS trend-line slopes were all computed from `student-por.csv` loaded reproducibly from the UCI archive.
- Figures saved to `outputs/`: `correlation_heatmap.png`, `scatter_trends.png`; tables saved as `descriptive_statistics.csv`, `pearson_spearman_vs_G3.csv`.
- Interpretation is derived from the computed correlation values above (no invented numbers).
